# LeetCode 2087 — Minimum Cost Homecoming of a Robot in a Grid bằng Python

Link đề: https://leetcode.com/problems/minimum-cost-homecoming-of-a-robot-in-a-grid/ · Độ khó: **Medium** · Chủ đề: Array, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minCost(self, startPos: List[int], homePos: List[int], rowCosts: List[int], colCosts: List[int]) -> int:
        ...
```

Lộ trình:
1. **Dijkstra trên lưới** (chỉ chạy được với lưới nhỏ) — coi bài như đường đi ngắn nhất tổng quát.
2. **Nhận xét "mỗi hàng ở giữa phải đi vào ít nhất một lần"** ⇒ cận dưới, và đường chữ L đạt đúng cận dưới.
3. **Cộng theo vòng lặp** từ hàng xuất phát tới hàng đích, cột tương tự — `O(m + n)`.
4. **Prefix sum** để trả lời nhiều truy vấn `(start, home)` trong `O(1)` mỗi truy vấn.

## 1. Mục tiêu học

1. Đọc kỹ cách tính chi phí: di chuyển **vào** ô ở hàng `r` theo chiều dọc tốn `rowCosts[r]`; **vào** ô ở cột `c` theo chiều ngang tốn `colCosts[c]`.
2. Chứng minh **cận dưới**: mọi đường đi phải "bước vào" từng hàng giữa xuất phát và đích ít nhất một lần — và đường đơn điệu đạt đúng cận đó.
3. Hiểu vì sao chi phí **không âm** là điều kiện then chốt (đi vòng không bao giờ có lợi).
4. Viết đúng chỉ số tổng đoạn cho cả hai chiều (đi xuống / đi lên), tránh lỗi off-by-one.
5. Biết khi nào phải quay lại thuật toán tổng quát (Dijkstra) — khi có vật cản hoặc chi phí phụ thuộc cả hàng lẫn cột.

## 2. Đề bài & ví dụ

Lưới `m × n`, ô `(0, 0)` ở góc trên trái. Robot ở `startPos = [sr, sc]`, nhà ở `homePos = [hr, hc]`. Mỗi bước đi sang một ô kề (lên/xuống/trái/phải), không ra ngoài lưới:

- Đi **lên hoặc xuống** vào ô thuộc hàng `r` tốn `rowCosts[r]`.
- Đi **trái hoặc phải** vào ô thuộc cột `c` tốn `colCosts[c]`.

Trả về tổng chi phí nhỏ nhất để robot về nhà.

**Ví dụ 1:** `startPos = [1, 0]`, `homePos = [2, 3]`, `rowCosts = [5, 4, 3]`, `colCosts = [8, 2, 6, 7]` → `18`: xuống `(2,0)` tốn `3`, sang phải `(2,1)`, `(2,2)`, `(2,3)` tốn `2 + 6 + 7`.

**Ví dụ 2:** `startPos = homePos = [0, 0]`, `rowCosts = [5]`, `colCosts = [26]` → `0`.

**Ràng buộc:** `1 ≤ m, n ≤ 10⁵`, `0 ≤ rowCosts[r], colCosts[c] ≤ 10⁴`.

In [1]:
import heapq
import random
import time
from itertools import accumulate
from typing import List

examples = [(([1, 0], [2, 3], [5, 4, 3], [8, 2, 6, 7]), 18),
            (([0, 0], [0, 0], [5], [26]), 0)]
for args, want in examples:
    print(args, "-> kỳ vọng", want)

([1, 0], [2, 3], [5, 4, 3], [8, 2, 6, 7]) -> kỳ vọng 18
([0, 0], [0, 0], [5], [26]) -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Tổng đoạn bằng lát cắt

`sum(a[l:r])` = `a[l] + … + a[r-1]` (không gồm `r`). Khi `l ≥ r` lát cắt rỗng, tổng `0` — tiện nhưng cũng dễ che giấu lỗi chiều đi.

In [2]:
a = [5, 4, 3]
print(sum(a[2:3]), sum(a[1:3]), sum(a[3:1]), "(lát cắt ngược chiều là rỗng)")

3 7 0 (lát cắt ngược chiều là rỗng)


### 3.2 Prefix sum

`P[i] = a[0] + … + a[i-1]` ⇒ `sum(a[l:r]) = P[r] - P[l]` trong `O(1)`.

In [3]:
P = [0] + list(accumulate(a))
print("P =", P, "| sum(a[1:3]) =", P[3] - P[1])

P = [0, 5, 9, 12] | sum(a[1:3]) = 7


### 3.3 Dijkstra với `heapq`

Đồ thị có trọng số **không âm** ⇒ Dijkstra cho đường ngắn nhất. Mỗi lần lấy đỉnh có khoảng cách nhỏ nhất ra khỏi heap; bỏ qua bản ghi cũ (`d > dist[u]`).

In [4]:
def dijkstra(adj, src):
    dist = {src: 0}; pq = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, w in adj.get(u, []):
            if d + w < dist.get(v, float("inf")):
                dist[v] = d + w; heapq.heappush(pq, (d + w, v))
    return dist

print(dijkstra({"A": [("B", 2), ("C", 5)], "B": [("C", 1)]}, "A"))

{'A': 0, 'B': 2, 'C': 3}


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: robot ở hàng `1`, nhà ở hàng `2`; cột `0` → cột `3`.

- Theo chiều dọc, robot phải **bước vào hàng 2** ít nhất một lần (không thì không bao giờ tới hàng 2) ⇒ tốn ít nhất `rowCosts[2] = 3`.
- Theo chiều ngang, phải bước vào cột `1`, `2`, `3` ít nhất mỗi cột một lần ⇒ ít nhất `2 + 6 + 7 = 15`.
- Cận dưới `3 + 15 = 18`, và đường "xuống rồi sang phải" đạt đúng `18`.

**Thứ tự không quan trọng:** đi sang phải trước rồi mới xuống cũng tốn `2 + 6 + 7 + 3 = 18`, vì chi phí chỉ phụ thuộc **hàng/cột được bước vào**, không phụ thuộc đang ở cột nào khi đi dọc.

| đường đi | các bước (chi phí) | tổng |
|---|---|---|
| xuống trước | ↓3, →2, →6, →7 | 18 |
| phải trước | →2, →6, →7, ↓3 | 18 |
| xen kẽ | →2, ↓3, →6, →7 | 18 |
| đi vòng (lên hàng 0 rồi xuống) | ↑5, →2, →6, →7, ↓4, ↓3 | 27 |

In [5]:
def path_cost(path, rowCosts, colCosts):
    total = 0
    for (r1, c1), (r2, c2) in zip(path, path[1:]):
        assert abs(r1 - r2) + abs(c1 - c2) == 1
        total += rowCosts[r2] if r1 != r2 else colCosts[c2]
    return total

R, C = [5, 4, 3], [8, 2, 6, 7]
paths = {
    "xuống trước": [(1, 0), (2, 0), (2, 1), (2, 2), (2, 3)],
    "phải trước": [(1, 0), (1, 1), (1, 2), (1, 3), (2, 3)],
    "xen kẽ": [(1, 0), (1, 1), (2, 1), (2, 2), (2, 3)],
    "đi vòng": [(1, 0), (0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3)],
}
for name, p in paths.items():
    print(f"{name:12s}: {path_cost(p, R, C)}")

xuống trước : 18
phải trước  : 18
xen kẽ      : 18
đi vòng     : 27


## 5. Cách 1 — Brute Force / baseline: Dijkstra trên lưới

**Ý tưởng:** coi mỗi ô là một đỉnh, mỗi bước là một cạnh có hướng với trọng số theo luật của đề, chạy Dijkstra từ `startPos`. Không cần bất kỳ nhận xét nào — dùng làm trọng tài.

In [6]:
def min_cost_dijkstra(startPos, homePos, rowCosts, colCosts):
    m, n = len(rowCosts), len(colCosts)
    INF = float("inf")
    dist = [[INF] * n for _ in range(m)]
    sr, sc = startPos
    dist[sr][sc] = 0
    pq = [(0, sr, sc)]
    while pq:
        d, r, c = heapq.heappop(pq)
        if d > dist[r][c]:
            continue
        if [r, c] == list(homePos):
            return d
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < m and 0 <= nc < n:
                w = rowCosts[nr] if nr != r else colCosts[nc]
                if d + w < dist[nr][nc]:
                    dist[nr][nc] = d + w
                    heapq.heappush(pq, (d + w, nr, nc))
    return dist[homePos[0]][homePos[1]]

for args, want in examples:
    print(min_cost_dijkstra(*args), "kỳ vọng", want)

18 kỳ vọng 18
0 kỳ vọng 0


**Độ phức tạp:** `O(m n log(m n))` thời gian, `O(m n)` bộ nhớ. Với `m = n = 10⁵` lưới có `10¹⁰` ô — **không thể** dựng.

**Nút thắt:** Dijkstra không khai thác cấu trúc đặc biệt: chi phí **tách được** theo hàng và cột.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (cận dưới):** giả sử `sr < hr`. Với mỗi hàng `r` thỏa `sr < r ≤ hr`, mọi đường đi từ hàng `sr` tới hàng `hr` phải có ít nhất một bước **dọc đi vào hàng `r`** (lần đầu tiên chỉ số hàng chạm `r`). Bước đó tốn `rowCosts[r]`. Các bước này khác nhau với các `r` khác nhau. Tương tự cho cột. Vì mọi chi phí `≥ 0`, các bước còn lại chỉ làm tăng tổng.

⇒ `chi phí ≥ Σ_{r giữa (sr, hr]} rowCosts[r] + Σ_{c giữa (sc, hc]} colCosts[c]`.

**Đạt được:** đường đơn điệu (chỉ đi về phía nhà) bước vào mỗi hàng/cột đó đúng một lần và không bước nào khác ⇒ tổng đúng bằng cận dưới.

**Quy tắc chỉ số:**
- *Nếu* `sr < hr` (đi xuống) *thì* cộng `rowCosts[sr+1 .. hr]`.
- *Nếu* `sr > hr` (đi lên) *thì* cộng `rowCosts[hr .. sr-1]` — các hàng **được bước vào** là `sr-1, sr-2, …, hr`.
- *Nếu* `sr = hr` *thì* `0`. Cột tương tự.

Nói gọn: **luôn bỏ hàng xuất phát, luôn tính hàng đích**.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng một đường chữ L

In [7]:
def min_cost_walk(startPos, homePos, rowCosts, colCosts):
    r, c = startPos
    hr, hc = homePos
    cost = 0
    while r != hr:                           # đi dọc trước
        r += 1 if hr > r else -1
        cost += rowCosts[r]                  # chi phí của hàng ĐƯỢC BƯỚC VÀO
    while c != hc:                           # rồi đi ngang
        c += 1 if hc > c else -1
        cost += colCosts[c]
    return cost

for args, want in examples:
    print(min_cost_walk(*args), "kỳ vọng", want)

18 kỳ vọng 18
0 kỳ vọng 0


**Độ phức tạp:** `O(|sr - hr| + |sc - hc|) = O(m + n)` thời gian, `O(1)` bộ nhớ. Ưu điểm: không phải nghĩ về chỉ số lát cắt — vòng lặp **tự** cộng đúng hàng được bước vào.

### 7.2 Cách 3 — Tổng đoạn bằng lát cắt (code nộp)

In [8]:
def min_cost_slices(startPos, homePos, rowCosts, colCosts):
    sr, sc = startPos
    hr, hc = homePos
    rows = sum(rowCosts[sr + 1:hr + 1]) if sr < hr else sum(rowCosts[hr:sr])
    cols = sum(colCosts[sc + 1:hc + 1]) if sc < hc else sum(colCosts[hc:sc])
    return rows + cols

for args, want in examples:
    print(min_cost_slices(*args), "kỳ vọng", want)

18 kỳ vọng 18
0 kỳ vọng 0


Kiểm tra chỉ số với chiều đi lên (`sr > hr`): các hàng bước vào là `hr, hr+1, …, sr-1` ⇒ `rowCosts[hr:sr]`. Khi `sr == hr`, nhánh `else` cho `rowCosts[hr:hr]` rỗng ⇒ `0`. ✓

Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** `O(m + n)` thời gian (tổng chạy trong C), `O(m + n)` bộ nhớ tạm cho lát cắt (có thể tránh bằng vòng lặp hoặc `itertools.islice`).

### 7.3 Cách 4 — Prefix sum cho nhiều truy vấn

Nếu cùng `rowCosts`, `colCosts` mà có nhiều cặp `(start, home)`, dựng prefix một lần rồi mỗi truy vấn `O(1)`.

In [9]:
class HomecomingOracle:
    def __init__(self, rowCosts, colCosts):
        self.PR = [0] + list(accumulate(rowCosts))
        self.PC = [0] + list(accumulate(colCosts))

    @staticmethod
    def _seg(P, s, h):
        # tổng chi phí các chỉ số được bước vào khi đi từ s tới h
        return P[h + 1] - P[s + 1] if s < h else P[s] - P[h]

    def cost(self, startPos, homePos):
        return self._seg(self.PR, startPos[0], homePos[0]) + self._seg(self.PC, startPos[1], homePos[1])

orc = HomecomingOracle([5, 4, 3], [8, 2, 6, 7])
print(orc.cost([1, 0], [2, 3]), orc.cost([2, 3], [1, 0]), orc.cost([0, 0], [0, 0]), orc.cost([2, 3], [0, 0]))

18 20 0 25


**Độ phức tạp:** dựng `O(m + n)`, mỗi truy vấn `O(1)`. Lưu ý `cost(A, B) ≠ cost(B, A)` nói chung: đi `A → B` trả tiền hàng/cột của **B và các hàng ở giữa**, đi ngược lại trả tiền của **A và các hàng ở giữa**.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1 và trên một trường hợp đi lên + sang trái.

In [10]:
def trace_walk(startPos, homePos, rowCosts, colCosts):
    r, c = startPos; hr, hc = homePos; cost = 0
    print(f"start = {startPos}, home = {homePos}")
    while r != hr:
        r += 1 if hr > r else -1; cost += rowCosts[r]
        print(f"  dọc  -> ({r},{c})  + rowCosts[{r}] = {rowCosts[r]:>2}  | tổng {cost}")
    while c != hc:
        c += 1 if hc > c else -1; cost += colCosts[c]
        print(f"  ngang-> ({r},{c})  + colCosts[{c}] = {colCosts[c]:>2}  | tổng {cost}")
    print("  =>", cost)
    return cost

trace_walk([1, 0], [2, 3], [5, 4, 3], [8, 2, 6, 7])
trace_walk([2, 3], [0, 1], [5, 4, 3], [8, 2, 6, 7])

start = [1, 0], home = [2, 3]
  dọc  -> (2,0)  + rowCosts[2] =  3  | tổng 3
  ngang-> (2,1)  + colCosts[1] =  2  | tổng 5
  ngang-> (2,2)  + colCosts[2] =  6  | tổng 11
  ngang-> (2,3)  + colCosts[3] =  7  | tổng 18
  => 18
start = [2, 3], home = [0, 1]
  dọc  -> (1,3)  + rowCosts[1] =  4  | tổng 4
  dọc  -> (0,3)  + rowCosts[0] =  5  | tổng 9
  ngang-> (0,2)  + colCosts[2] =  6  | tổng 15
  ngang-> (0,1)  + colCosts[1] =  2  | tổng 17
  => 17


17

Bảng cho trường hợp thứ hai (`(2,3) → (0,1)`): hàng được bước vào là `1, 0` (`4 + 5 = 9`), cột được bước vào là `2, 1` (`6 + 2 = 8`) ⇒ `17`. Theo lát cắt: `rowCosts[0:2] = [5, 4]`, `colCosts[1:3] = [2, 6]` ✓.

In [11]:
def trace_dijkstra_order(startPos, homePos, rowCosts, colCosts, limit=12):
    m, n = len(rowCosts), len(colCosts); INF = float("inf")
    dist = [[INF] * n for _ in range(m)]; sr, sc = startPos; dist[sr][sc] = 0; pq = [(0, sr, sc)]; k = 0
    while pq and k < limit:
        d, r, c = heapq.heappop(pq)
        if d > dist[r][c]: continue
        k += 1; print(f"  lấy ra ({r},{c}) với d = {d}")
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < m and 0 <= nc < n:
                w = rowCosts[nr] if nr != r else colCosts[nc]
                if d + w < dist[nr][nc]: dist[nr][nc] = d + w; heapq.heappush(pq, (d + w, nr, nc))
    print("  bảng dist (inf = chưa tới):")
    for row in dist: print("   ", ["∞" if x == INF else x for x in row])

trace_dijkstra_order([1, 0], [2, 3], [5, 4, 3], [8, 2, 6, 7], limit=20)

  lấy ra (1,0) với d = 0
  lấy ra (1,1) với d = 2
  lấy ra (2,0) với d = 3
  lấy ra (0,0) với d = 5
  lấy ra (2,1) với d = 5
  lấy ra (0,1) với d = 7
  lấy ra (1,2) với d = 8
  lấy ra (2,2) với d = 11
  lấy ra (0,2) với d = 13
  lấy ra (1,3) với d = 15
  lấy ra (2,3) với d = 18
  lấy ra (0,3) với d = 20
  bảng dist (inf = chưa tới):
    [5, 7, 13, 20]
    [0, 2, 8, 15]
    [3, 5, 11, 18]


Bảng `dist` của Dijkstra cho thấy: giá trị tại mỗi ô `(r, c)` đúng bằng công thức "hàng được bước vào + cột được bước vào" từ `(1, 0)` — ví dụ `dist[0][3] = 5 + 2 + 6 + 7 = 20`.

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def minCost(self, startPos: List[int], homePos: List[int], rowCosts: List[int], colCosts: List[int]) -> int:
        sr, sc = startPos
        hr, hc = homePos
        if sr < hr:
            rows = sum(rowCosts[sr + 1:hr + 1])     # đi xuống: vào các hàng sr+1..hr
        else:
            rows = sum(rowCosts[hr:sr])             # đi lên: vào các hàng sr-1..hr
        if sc < hc:
            cols = sum(colCosts[sc + 1:hc + 1])
        else:
            cols = sum(colCosts[hc:sc])
        return rows + cols

sol = Solution()
for args, want in examples:
    got = sol.minCost(*args)
    print(got, "OK" if got == want else "SAI")

18 OK
0 OK


## 10. Vì sao thuật toán đúng

Giả sử `sr ≤ hr` (trường hợp `sr > hr` đối xứng). Xét một đường đi bất kỳ `P` từ start tới home.

**Bổ đề 1 (mỗi hàng giữa phải được bước vào).** Với mỗi `r` thỏa `sr < r ≤ hr`: chỉ số hàng trên `P` bắt đầu ở `sr < r` và kết thúc ở `hr ≥ r`; mỗi bước dọc đổi hàng đúng `1` ⇒ tồn tại một bước đầu tiên đi từ hàng `r - 1` vào hàng `r`. Bước đó tốn `rowCosts[r]`.

**Bổ đề 2 (các bước đó khác nhau).** Bước "vào hàng `r` lần đầu" với các `r` khác nhau là các bước khác nhau của `P`; tương tự các bước "vào cột `c` lần đầu"; bước dọc và bước ngang cũng khác nhau.

⇒ `cost(P) ≥ Σ_{sr < r ≤ hr} rowCosts[r] + Σ_{cột giữa} colCosts[c]`, vì các bước khác có chi phí `≥ 0`.

**Đạt cận.** Đường chữ L của Cách 2 gồm đúng các bước trong Bổ đề 1 (mỗi hàng một lần) và các bước cột tương ứng — không bước nào khác ⇒ chi phí bằng cận dưới. ✓

**Vai trò của chi phí không âm:** nếu cho phép chi phí âm, đi qua lại giữa hai hàng có chi phí âm sẽ làm tổng giảm vô hạn — bài không còn đáp án hữu hạn.

## 11. Phân tích Time / Space

`m = len(rowCosts)`, `n = len(colCosts)`, đều `≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Dijkstra lưới | `O(mn log(mn))` | `O(mn)` |
| 2. Đi chữ L | `O(m + n)` | `O(1)` |
| 3. Lát cắt + `sum` | `O(m + n)` | `O(m + n)` tạm |
| 4. Prefix (nhiều truy vấn) | dựng `O(m + n)`, truy vấn `O(1)` | `O(m + n)` |

Kết quả tối đa: `(10⁵ + 10⁵) · 10⁴ = 2·10⁹` — vẫn vừa `int32` (`2 147 483 647`) nhưng rất sát; ở C++/Java nên dùng `long` cho an toàn (Python không giới hạn).

In [13]:
rng = random.Random(1)
m = n = 10**5
R = [rng.randint(0, 10**4) for _ in range(m)]; Cc = [rng.randint(0, 10**4) for _ in range(n)]
cases = [([0, 0], [m - 1, n - 1]), ([m - 1, n - 1], [0, 0]), ([50000, 3], [50000, 99999])]
for name, fn in [("walk", min_cost_walk), ("slices", min_cost_slices), ("submit", Solution().minCost)]:
    t0 = time.perf_counter(); res = [fn(s, h, R, Cc) for s, h in cases]
    print(f"{name:7s}: {res}  {(time.perf_counter() - t0) * 1000:.1f} ms")
orc = HomecomingOracle(R, Cc)
t0 = time.perf_counter(); qs = [orc.cost([rng.randrange(m), rng.randrange(n)], [rng.randrange(m), rng.randrange(n)]) for _ in range(10**5)]
print(f"oracle: 1e5 truy vấn {time.perf_counter() - t0:.2f}s")
print("max có thể:", (m + n) * 10**4, "<= 2^31 - 1 =", 2**31 - 1, ":", (m + n) * 10**4 <= 2**31 - 1)

walk   : [1000445036, 1000437337, 499441407]  22.2 ms
slices : [1000445036, 1000437337, 499441407]  4.7 ms
submit : [1000445036, 1000437337, 499441407]  4.3 ms


oracle: 1e5 truy vấn 0.13s
max có thể: 2000000000 <= 2^31 - 1 = 2147483647 : True


## 12. Bộ test

In [14]:
fns = [min_cost_walk, min_cost_slices, lambda s, h, r, c: HomecomingOracle(r, c).cost(s, h), Solution().minCost]

for args, want in examples:
    assert min_cost_dijkstra(*args) == want
    for fn in fns:
        assert fn(*args) == want

edge = [
    (([0, 0], [0, 0], [1], [1]), 0),
    (([0, 0], [0, 1], [7], [3, 9]), 9),                 # một bước ngang
    (([0, 1], [0, 0], [7], [3, 9]), 3),                 # ngược lại: trả cột 0
    (([2, 0], [0, 0], [1, 2, 3], [5]), 1 + 2),          # đi lên: hàng 1, 0
    (([0, 0], [2, 2], [0, 0, 0], [0, 0, 0]), 0),        # chi phí 0
    (([1, 1], [1, 1], [10**4] * 3, [10**4] * 3), 0),
]
for args, want in edge:
    assert min_cost_dijkstra(*args) == want, args
    for fn in fns:
        assert fn(*args) == want, (fn, args)

rng = random.Random(2024)
for _ in range(500):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    R = [rng.randint(0, 20) for _ in range(m)]; Cc = [rng.randint(0, 20) for _ in range(n)]
    s = [rng.randrange(m), rng.randrange(n)]; h = [rng.randrange(m), rng.randrange(n)]
    want = min_cost_dijkstra(s, h, R, Cc)
    for fn in fns:
        assert fn(s, h, R, Cc) == want, (fn, s, h, R, Cc)
print("ví dụ + edge + 500 lưới ngẫu nhiên khớp Dijkstra")

ví dụ + edge + 500 lưới ngẫu nhiên khớp Dijkstra


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tính cả hàng/cột xuất phát

In [15]:
def wrong_include_start(startPos, homePos, rowCosts, colCosts):
    sr, sc = startPos; hr, hc = homePos
    rows = sum(rowCosts[min(sr, hr):max(sr, hr) + 1])        # SAI: gồm cả hàng xuất phát
    cols = sum(colCosts[min(sc, hc):max(sc, hc) + 1])
    return rows + cols

for args, want in examples:
    print("sai =", wrong_include_start(*args), "| đúng =", want)

sai = 30 | đúng = 18
sai = 31 | đúng = 0


Robot **đã** ở hàng xuất phát — không bước vào nó. Ví dụ 1: cộng thêm `rowCosts[1] = 4` và `colCosts[0] = 8`; ví dụ 2: cộng `5 + 26` dù không đi bước nào. **Sửa:** bỏ hàng/cột xuất phát, tính hàng/cột đích.

### Lỗi 2 — Đi lên nhưng dùng công thức của đi xuống (lấy `min`/`max` cho gọn)

In [16]:
def wrong_symmetric(startPos, homePos, rowCosts, colCosts):
    sr, sc = startPos; hr, hc = homePos
    lo_r, hi_r = min(sr, hr), max(sr, hr)
    lo_c, hi_c = min(sc, hc), max(sc, hc)
    return sum(rowCosts[lo_r + 1:hi_r + 1]) + sum(colCosts[lo_c + 1:hi_c + 1])   # SAI khi đi lên/trái

R, C = [5, 4, 3], [8, 2, 6, 7]
for s, h in [([1, 0], [2, 3]), ([2, 3], [1, 0]), ([2, 3], [0, 1])]:
    print(f"{s} -> {h}: sai = {wrong_symmetric(s, h, R, C)} | đúng = {min_cost_dijkstra(s, h, R, C)}")

[1, 0] -> [2, 3]: sai = 18 | đúng = 18
[2, 3] -> [1, 0]: sai = 18 | đúng = 20
[2, 3] -> [0, 1]: sai = 20 | đúng = 17


Khi đi lên, hàng **được bước vào** là `hr..sr-1` (gồm hàng đích, bỏ hàng xuất phát), không phải `hr+1..sr`. "Đối xứng hóa" bằng `min/max` đổi vai trò start/home. **Sửa:** tách hai nhánh như Cách 3, hoặc mô phỏng bằng vòng lặp (Cách 2).

### Lỗi 3 — Cộng chi phí ô **rời đi** thay vì ô **bước vào**

In [17]:
def wrong_leaving_cost(startPos, homePos, rowCosts, colCosts):
    r, c = startPos; hr, hc = homePos; cost = 0
    while r != hr:
        cost += rowCosts[r]; r += 1 if hr > r else -1          # SAI: cộng trước khi bước
    while c != hc:
        cost += colCosts[c]; c += 1 if hc > c else -1
    return cost

for args, want in examples:
    print("sai =", wrong_leaving_cost(*args), "| đúng =", want)

sai = 20 | đúng = 18
sai = 0 | đúng = 0


**Sửa:** bước trước, cộng chi phí ô mới sau.

### Lỗi 4 — Dùng một lát cắt cho mọi chiều: `sum(rowCosts[sr+1:hr+1])`

In [18]:
def wrong_one_direction(startPos, homePos, rowCosts, colCosts):
    sr, sc = startPos; hr, hc = homePos
    return sum(rowCosts[sr + 1:hr + 1]) + sum(colCosts[sc + 1:hc + 1])   # SAI: rỗng khi đi lên/trái

for s, h in [([1, 0], [2, 3]), ([2, 3], [0, 1]), ([2, 0], [0, 0])]:
    print(f"{s} -> {h}: sai = {wrong_one_direction(s, h, R, C)} | đúng = {min_cost_dijkstra(s, h, R, C)}")

[1, 0] -> [2, 3]: sai = 18 | đúng = 18
[2, 3] -> [0, 1]: sai = 0 | đúng = 17
[2, 0] -> [0, 0]: sai = 0 | đúng = 9


Lát cắt "ngược chiều" trong Python **không báo lỗi** mà âm thầm trả danh sách rỗng ⇒ tổng `0`. Đây là lỗi khó thấy nếu chỉ test với ví dụ đề (đều đi xuống/phải). **Sửa:** xử lý riêng `sr > hr`, `sc > hc` và luôn test cả bốn hướng.

### Lỗi 5 — Nghĩ rằng phải chọn thứ tự "dọc trước" hay "ngang trước" (rồi lấy `min`)

Không sai kết quả nhưng thừa: mọi đường đơn điệu có **cùng** chi phí (mục 4). Hiểu điều này giúp tránh viết DP/Dijkstra không cần thiết.

In [19]:
def l_path_cost(startPos, homePos, rowCosts, colCosts, vertical_first):
    r, c = startPos; hr, hc = homePos; cost = 0
    order = ["v", "h"] if vertical_first else ["h", "v"]
    for d in order:
        if d == "v":
            while r != hr: r += 1 if hr > r else -1; cost += rowCosts[r]
        else:
            while c != hc: c += 1 if hc > c else -1; cost += colCosts[c]
    return cost

rng = random.Random(5)
for _ in range(1000):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    Rr = [rng.randint(0, 9) for _ in range(m)]; Cr = [rng.randint(0, 9) for _ in range(n)]
    s = [rng.randrange(m), rng.randrange(n)]; h = [rng.randrange(m), rng.randrange(n)]
    assert l_path_cost(s, h, Rr, Cr, True) == l_path_cost(s, h, Rr, Cr, False)
print("dọc-trước và ngang-trước luôn cùng chi phí (1000 bộ ngẫu nhiên)")

dọc-trước và ngang-trước luôn cùng chi phí (1000 bộ ngẫu nhiên)


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Chi phí **tách được** (phụ thuộc riêng hàng, riêng cột) + không âm ⇒ tách bài 2D thành hai bài 1D.
- "Mỗi mức giữa phải được đi qua ít nhất một lần" là kỹ thuật **cận dưới** kinh điển; khi một đường cụ thể đạt cận ⇒ xong.
- Với tổng đoạn có chiều: viết hai nhánh rõ ràng hoặc mô phỏng; test cả 4 hướng đi.
- Nếu có vật cản / chi phí theo ô ⇒ quay về Dijkstra / 0-1 BFS.
- Họ bài: 62 *Unique Paths*, 64 *Minimum Path Sum* (chỉ đi phải/xuống, chi phí theo ô ⇒ DP), 1631 *Path With Minimum Effort*, 2290 *Minimum Obstacle Removal to Reach Corner*.

## 15. Biến thể của bài

1. **Nhiều truy vấn** trên cùng lưới: prefix sum (Cách 4).
2. **Chi phí phụ thuộc chiều đi** (`upCosts`, `downCosts`, `leftCosts`, `rightCosts`): vẫn tách được — chọn mảng theo chiều.
3. **Chi phí theo ô** `cost[r][c]` khi bước vào: không tách được ⇒ Dijkstra (hoặc DP nếu chỉ đi phải/xuống).
4. **Có vật cản**: đường đơn điệu có thể bị chặn; công thức chỉ còn là **cận dưới** ⇒ Dijkstra (Bài 5).
5. **Chi phí âm**: đáp án `-∞` nếu có thể đi qua lại giữa hai ô kề mà tổng hai bước âm.

In [20]:
def directional_cost(startPos, homePos, up, down, left, right):
    """Biến thể 2: bước vào hàng r từ trên xuống tốn down[r], từ dưới lên tốn up[r]; tương tự cột."""
    sr, sc = startPos; hr, hc = homePos
    rows = sum(down[sr + 1:hr + 1]) if sr < hr else sum(up[hr:sr])
    cols = sum(right[sc + 1:hc + 1]) if sc < hc else sum(left[hc:sc])
    return rows + cols

def directional_dijkstra(startPos, homePos, up, down, left, right):
    m, n = len(up), len(left); INF = float("inf")
    dist = {tuple(startPos): 0}; pq = [(0, *startPos)]
    while pq:
        d, r, c = heapq.heappop(pq)
        if d > dist[(r, c)]: continue
        for nr, nc, w in ((r + 1, c, down), (r - 1, c, up), (r, c + 1, right), (r, c - 1, left)):
            if 0 <= nr < m and 0 <= nc < n:
                cost = w[nr] if nr != r else w[nc]
                if d + cost < dist.get((nr, nc), INF):
                    dist[(nr, nc)] = d + cost; heapq.heappush(pq, (d + cost, nr, nc))
    return dist[tuple(homePos)]

rng = random.Random(15)
for _ in range(300):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    arrs = [[rng.randint(0, 9) for _ in range(k)] for k in (m, m, n, n)]
    s = [rng.randrange(m), rng.randrange(n)]; h = [rng.randrange(m), rng.randrange(n)]
    assert directional_cost(s, h, *arrs) == directional_dijkstra(s, h, *arrs)
print("biến thể chi phí theo chiều: công thức tách được khớp Dijkstra (biến thể tự đặt)")

biến thể chi phí theo chiều: công thức tách được khớp Dijkstra (biến thể tự đặt)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dijkstra | đường ngắn nhất tổng quát | `O(mn log mn)` | `O(mn)` | kiểm chứng, có vật cản |
| 2. Đi chữ L | mô phỏng | `O(m + n)` | `O(1)` | ít lỗi chỉ số |
| 3. Lát cắt + `sum` | công thức | `O(m + n)` | `O(m + n)` | **nộp bài** |
| 4. Prefix sum | tổng đoạn `O(1)` | `O(1)`/truy vấn | `O(m + n)` | nhiều truy vấn |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** với `rowCosts = [5, 4, 3]`, `colCosts = [8, 2, 6, 7]`: `(2,3) → (1,0)`? `(0,0) → (2,3)`? `(1,2) → (1,2)`?

**Bài 2 (tính tay):** `startPos = [3, 1]`, `homePos = [0, 4]`, `rowCosts = [1, 2, 3, 4]`, `colCosts = [9, 8, 7, 6, 5]`. Liệt kê hàng và cột được bước vào, tính tổng.

**Bài 3 (điền TODO):** hoàn thiện `seg_cost(costs, s, h)` trả về tổng chi phí các chỉ số được bước vào khi đi từ `s` tới `h` trên một trục.

**Bài 4 (tự viết lại):** viết `round_trip(startPos, homePos, rowCosts, colCosts)` = chi phí đi về nhà **rồi quay lại** chỗ cũ. Chứng minh/kiểm chứng rằng nó bằng `Σ(các hàng từ min tới max) − …`: tìm công thức theo `rowCosts[sr]`, `rowCosts[hr]` và tổng đoạn.

**Bài 5 (mini challenge):** lưới có **vật cản** `blocked[r][c] = True` (robot không được vào). Chi phí di chuyển vẫn theo luật đề. Viết Dijkstra trả về chi phí nhỏ nhất hoặc `-1`; kiểm chứng rằng khi không có vật cản kết quả bằng công thức, và khi có vật cản kết quả **không nhỏ hơn** công thức. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [21]:
def seg_cost(costs, s, h):
    # TODO: s < h -> sum(costs[s+1:h+1]); s > h -> sum(costs[h:s]); s == h -> 0
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Áp quy tắc "bỏ xuất phát, tính đích". (2) `(2,3) → (1,0)`: hàng 1 (`4`), cột 2, 1, 0 (`6 + 2 + 8`). (3) Kết quả `20`; `(0,0) → (2,3)`: `4 + 3 + 2 + 6 + 7 = 22`; cùng ô ⇒ `0`.

**Bài 2:** (1) Đi lên: hàng `2, 1, 0`. (2) Đi phải: cột `2, 3, 4`. (3) `(3 + 2 + 1) + (7 + 6 + 5) = 24`.

**Bài 3:** (1) Ba nhánh. (2) Có thể viết `sum(costs[s+1:h+1]) if s < h else sum(costs[h:s])`. (3) Kiểm tra với vòng lặp mô phỏng một trục.

**Bài 4:** (1) Chi phí đi = hàng giữa + hàng đích; chi phí về = hàng giữa + hàng xuất phát. (2) Tổng một trục = `sum(costs[lo:hi+1]) + sum(costs[lo+1:hi])` khi `lo < hi` (mọi hàng trong đoạn đóng, cộng thêm các hàng ở giữa lần nữa). (3) Viết gọn: `2·sum(costs[lo:hi+1]) − costs[lo] − costs[hi]`.

**Bài 5:** (1) Dùng Dijkstra của Cách 1, bỏ qua ô bị chặn. (2) Nếu nhà không tới được ⇒ `-1`. (3) Với vật cản, mọi đường vẫn phải bước vào các hàng/cột giữa ⇒ công thức là **cận dưới**.

## 19. Đáp án tham khảo

In [22]:
R, C = [5, 4, 3], [8, 2, 6, 7]
# Bài 1
assert [Solution().minCost(s, h, R, C) for s, h in [([2, 3], [1, 0]), ([0, 0], [2, 3]), ([1, 2], [1, 2])]] == [20, 22, 0]
# Bài 2
assert Solution().minCost([3, 1], [0, 4], [1, 2, 3, 4], [9, 8, 7, 6, 5]) == 24 == min_cost_dijkstra([3, 1], [0, 4], [1, 2, 3, 4], [9, 8, 7, 6, 5])

# Bài 3
def seg_cost_ans(costs, s, h):
    if s < h: return sum(costs[s + 1:h + 1])
    if s > h: return sum(costs[h:s])
    return 0

def seg_walk(costs, s, h):
    t = 0
    while s != h:
        s += 1 if h > s else -1; t += costs[s]
    return t

rng = random.Random(31)
for _ in range(500):
    costs = [rng.randint(0, 9) for _ in range(rng.randint(1, 10))]
    s, h = rng.randrange(len(costs)), rng.randrange(len(costs))
    assert seg_cost_ans(costs, s, h) == seg_walk(costs, s, h)

# Bài 4
def round_trip_ans(startPos, homePos, rowCosts, colCosts):
    return Solution().minCost(startPos, homePos, rowCosts, colCosts) + Solution().minCost(homePos, startPos, rowCosts, colCosts)

def round_trip_formula(startPos, homePos, rowCosts, colCosts):
    def axis(costs, a, b):
        lo, hi = min(a, b), max(a, b)
        return 0 if lo == hi else 2 * sum(costs[lo:hi + 1]) - costs[lo] - costs[hi]
    return axis(rowCosts, startPos[0], homePos[0]) + axis(colCosts, startPos[1], homePos[1])

for _ in range(500):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    Rr = [rng.randint(0, 9) for _ in range(m)]; Cr = [rng.randint(0, 9) for _ in range(n)]
    s = [rng.randrange(m), rng.randrange(n)]; h = [rng.randrange(m), rng.randrange(n)]
    assert round_trip_ans(s, h, Rr, Cr) == round_trip_formula(s, h, Rr, Cr)
print("Bài 4 ví dụ 1 (đi và về):", round_trip_ans([1, 0], [2, 3], R, C), "= 18 +", Solution().minCost([2, 3], [1, 0], R, C))
print("Bài 1-4: OK")

Bài 4 ví dụ 1 (đi và về): 38 = 18 + 20
Bài 1-4: OK


In [23]:
# Bài 5 — vật cản, Dijkstra
def min_cost_blocked(startPos, homePos, rowCosts, colCosts, blocked):
    m, n = len(rowCosts), len(colCosts); INF = float("inf")
    sr, sc = startPos
    if blocked[sr][sc] or blocked[homePos[0]][homePos[1]]:
        return -1
    dist = [[INF] * n for _ in range(m)]; dist[sr][sc] = 0; pq = [(0, sr, sc)]
    while pq:
        d, r, c = heapq.heappop(pq)
        if d > dist[r][c]: continue
        if (r, c) == tuple(homePos): return d
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < m and 0 <= nc < n and not blocked[nr][nc]:
                w = rowCosts[nr] if nr != r else colCosts[nc]
                if d + w < dist[nr][nc]: dist[nr][nc] = d + w; heapq.heappush(pq, (d + w, nr, nc))
    return -1

rng = random.Random(55)
detour = None
for _ in range(400):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    Rr = [rng.randint(0, 9) for _ in range(m)]; Cr = [rng.randint(0, 9) for _ in range(n)]
    s = [rng.randrange(m), rng.randrange(n)]; h = [rng.randrange(m), rng.randrange(n)]
    free = [[False] * n for _ in range(m)]
    assert min_cost_blocked(s, h, Rr, Cr, free) == Solution().minCost(s, h, Rr, Cr)
    blk = [[rng.random() < 0.25 for _ in range(n)] for _ in range(m)]
    blk[s[0]][s[1]] = blk[h[0]][h[1]] = False
    got = min_cost_blocked(s, h, Rr, Cr, blk)
    assert got == -1 or got >= Solution().minCost(s, h, Rr, Cr)
    if detour is None and got > Solution().minCost(s, h, Rr, Cr):
        detour = (s, h, Rr, Cr, blk, got, Solution().minCost(s, h, Rr, Cr))
print("Bài 5: không vật cản = công thức; có vật cản >= công thức")
if detour:
    s, h, Rr, Cr, blk, got, lb = detour
    print("ví dụ phải đi vòng:", s, "->", h, "| chi phí", got, "> cận dưới", lb)
    for r in range(len(Rr)):
        print("   ", "".join("#" if blk[r][c] else ("S" if [r, c] == s else ("H" if [r, c] == h else ".")) for c in range(len(Cr))))

Bài 5: không vật cản = công thức; có vật cản >= công thức
ví dụ phải đi vòng: [1, 2] -> [0, 1] | chi phí 37 > cận dưới 12
    .H#.#
    .#S..
    ...#.
    ....#


## 20. Template nhớ nhanh + Checklist

```python
# Chi phí tách được theo trục: BỎ chỉ số xuất phát, TÍNH chỉ số đích
def axis(costs, s, h):
    if s < h: return sum(costs[s + 1:h + 1])   # đi tăng
    return sum(costs[h:s])                     # đi giảm (hoặc s == h -> rỗng -> 0)

ans = axis(rowCosts, sr, hr) + axis(colCosts, sc, hc)
```

**Checklist trước khi submit:**
- [ ] Không tính hàng/cột xuất phát; có tính hàng/cột đích.
- [ ] Xử lý riêng chiều đi lên/trái (lát cắt ngược chiều âm thầm rỗng).
- [ ] Test đủ 4 hướng và trường hợp start == home.
- [ ] Không cần chọn thứ tự dọc/ngang — mọi đường đơn điệu cùng chi phí.
- [ ] Vật cản hoặc chi phí theo ô ⇒ đổi sang Dijkstra.